## Novel model approach

This notebook serves as the source code for all the model testing and training (along with hyperparam grid search) before the development/submission of the final best model. This model approach tries a variation on the transformer architecture, with different heads, as detailed in the report.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, average_precision_score

In [2]:
from pcl_tf.dataset_manager import DatasetManager as DM
from pcl_tf.collation import collate_fn

In [3]:
NUM_LABELS = 7
LOAD_BATCH_SIZE = 16
ACCUM_STEPS = 3  # effective batch size = LOAD_BATCH_SIZE * ACCUM_STEPS = 32
LOCAL_CACHE_DIR = './models_cache'
MODEL_NAME = "albert-base-v2"
NUM_WORKERS = 0
PIN_MEMORY = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
train_labels_path = "data/train_semeval_parids-labels.csv"
dev_labels_path = "data/dev_semeval_parids-labels.csv"
texts_path = "data/dontpatronizeme_pcl_cleaned.csv"
test_path = "data/task4_test.tsv"
cats_path = "data/dontpatronizeme_categories.tsv"

In [5]:
texts_df = pd.read_csv(texts_path, low_memory=False)
texts_df["par_id"] = texts_df["par_id"].astype(int)
texts_df = texts_df.set_index("par_id")

In [6]:
from pcl_tf.feature_engineering import build_auxiliary_features, transform_auxiliary_features

train_labels = pd.read_csv(train_labels_path)
train_par_ids = train_labels["par_id"].astype(int).values
train_texts = texts_df.loc[train_par_ids, "text"]

aux_train, aux_meta = build_auxiliary_features(train_texts, ngram_range=(1, 3),
                                                max_features=200, min_df=5)
AUX_DIM = aux_meta["total_dim"]
print(f"Auxiliary feature dim: {AUX_DIM} (NER={aux_meta['n_ner']}, ngram={aux_meta['n_ngram']})")

Auxiliary feature dim: 218 (NER=18, ngram=200)


In [7]:
dev_labels = pd.read_csv(dev_labels_path)
dev_par_ids = dev_labels["par_id"].astype(int).values
dev_par_ids = dev_par_ids[np.isin(dev_par_ids, texts_df.index)]
dev_texts = texts_df.loc[dev_par_ids, "text"]
aux_dev = transform_auxiliary_features(dev_texts, aux_meta)

In [8]:
training_ds = DM(train_labels_path, texts_df=texts_df, aux_features=aux_train)
training_ds.print_stats()

Total samples: 8375
Binary distribution: [7581  794]
Multilabel distribution: [574. 160. 162. 192. 145. 363.  29.]


In [9]:
dev_ds = DM(dev_labels_path, texts_df=texts_df, aux_features=aux_dev)
dev_ds.print_stats()

Total samples: 2093
Binary distribution: [1894  199]
Multilabel distribution: [142.  36.  62.  38.  52. 106.  11.]


In [10]:
def collate_fn_wrapper(tokenizer):
    def collate_fn_inner(batch):
        return collate_fn(tokenizer, batch)
    return collate_fn_inner

In [11]:
def evaluate_dev(model, dataloader, device):
    """Evaluate model on dev set. Primary metric: F1 of positive (PCL) class."""
    model.eval()
    bin_probs=[]
    bin_labels=[]
    multi_probs=[]
    multi_labels=[]

    with torch.no_grad():
        for b in dataloader:
            input_ids = b["input_ids"].to(device)
            attention_mask = b["attention_mask"].to(device)
            labels = b["labels"].to(device)
            aux_features = b["aux_features"].to(device) if "aux_features" in b else None

            out = model(input_ids=input_ids, attention_mask=attention_mask, aux_features=aux_features)
            
            bin_probs.append(torch.sigmoid(out["logit_bin"]).cpu().numpy())
            multi_probs.append(torch.sigmoid(out["logit_multi"]).cpu().numpy())
            
            bin_labels.append(labels[:,0].cpu().numpy())
            multi_labels.append(labels[:,1:].cpu().numpy())
            
    bin_probs = np.concatenate(bin_probs)
    bin_labels = np.concatenate(bin_labels)
    multi_probs = np.concatenate(multi_probs)
    multi_labels = np.concatenate(multi_labels)

    # Primary task metric: F1 of positive (PCL) class
    bin_preds = (bin_probs >= 0.5).astype(int)
    bin_f1 = f1_score(bin_labels, bin_preds, pos_label=1, zero_division=0)

    # Secondary diagnostics
    multi_micro_f1 = f1_score(multi_labels.flatten(), (multi_probs >= 0.1).flatten(), zero_division=0)
    bin_ap = average_precision_score(bin_labels, bin_probs)

    return {"bin_f1": bin_f1, "multi_micro_f1": multi_micro_f1, "bin_ap": bin_ap}

In [12]:
import optuna
import pcl_tf.collation as pcl_collation
from pcl_tf.tf import PCLModel, get_tokenizer

scaler = torch.amp.GradScaler("cuda")  # for mixed-precision training

def objective(trial):
    model_name = trial.suggest_categorical("model_name", ["albert-base-v2", "microsoft/deberta-v3-small"])
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    wd = trial.suggest_float("wd", 1e-4, 1e-2, log=True)
    max_len = trial.suggest_categorical("max_len", [128, 256])
    dropout = 0 # any other dropout value causes grad explosion prob due to small batch size + complex task
    epochs = trial.suggest_int("epochs", 3, 12)

    trial_tokenizer = get_tokenizer(model_name)

    pcl_collation.MAX_LEN = max_len

    trial_train_loader = DataLoader(
        training_ds,
        batch_size=LOAD_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_wrapper(trial_tokenizer),
        pin_memory=PIN_MEMORY,
        num_workers=NUM_WORKERS,
    )

    trial_dev_loader = DataLoader(
        dev_ds,
        batch_size=LOAD_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_wrapper(trial_tokenizer),
        pin_memory=PIN_MEMORY,
        num_workers=NUM_WORKERS,
    )

    model = PCLModel(model_name, n_labels=NUM_LABELS, aux_dim=AUX_DIM, dropout=dropout, device=DEVICE).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    print("Config:", {"model_name": model_name, "lr": lr, "wd": wd, "max_len": max_len, "dropout": dropout, "epochs": epochs, "accum_steps": ACCUM_STEPS})

    try:
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            optim.zero_grad()

            for step, batch in enumerate(trial_train_loader):
                input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
                attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
                labels = batch["labels"].to(DEVICE, non_blocking=True)
                aux_features = batch["aux_features"].to(DEVICE, non_blocking=True) if "aux_features" in batch else None

                with torch.amp.autocast("cuda"): # fp16 to halve mem usage to avoid oom
                    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, aux_features=aux_features)
                    loss = out["loss"] / ACCUM_STEPS

                scaler.scale(loss).backward()
                running_loss += out["loss"].item()

                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(trial_train_loader): # handle odd cases asw
                    scaler.step(optim)
                    scaler.update()
                    optim.zero_grad()

            print(f"Epoch {epoch + 1} - Average Loss: {running_loss / len(trial_train_loader)}")

    except torch.cuda.OutOfMemoryError:
        print("OOM: pruning this trial")
        torch.cuda.empty_cache()
        raise optuna.TrialPruned()

    metrics = evaluate_dev(model, trial_dev_loader, DEVICE)
    print("Trial metrics:", str(metrics))
    del model
    torch.cuda.empty_cache()
    return metrics["bin_f1"]  # optimize for F1 of positive (PCL) class — the actual task metric

In [13]:
torch.cuda.empty_cache()

In [ ]:
study = optuna.create_study(direction="maximize", study_name="pcl_hyperparam_search")
study.optimize(objective, n_trials=50, n_jobs=1)

In [ ]:
best_trial = study.best_trial
print("Best trial:")
print(f"  Value: {best_trial.value}")
print("  Params:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
results = []
for trial in study.trials:
    results.append({**trial.params, "value": trial.value})
res_df = pd.DataFrame(results)
res_df.to_csv("optuna_results.csv", index=False)
print("Saved optuna_results.csv")

In [14]:
import os

prior_results_path = "optuna_results.csv"
prior_df = pd.read_csv(prior_results_path)

if "value" not in prior_df.columns:
    raise ValueError("Expected a 'value' column in optuna_results.csv")

best_idx = prior_df["value"].astype(float).idxmax()
best_prior_trial = prior_df.loc[best_idx].to_dict()

base_best_params = {
    "model_name": str(best_prior_trial["model_name"]),
    "lr": float(best_prior_trial["lr"]),
    "wd": float(best_prior_trial["wd"]),
    "max_len": int(best_prior_trial["max_len"]),
    "epochs": int(best_prior_trial["epochs"]),
    "dropout": 0.0,
}

print("Best prior trial from optuna_results.csv:")
print(base_best_params)
print(f"best_f1={float(best_prior_trial['value']):.6f}")

Best prior trial from optuna_results.csv:
{'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'epochs': 7, 'dropout': 0.0}
best_f1=0.541966


In [15]:
refined_results_csv = "pcl_refined_hyperparam_results.csv"

if os.path.exists(refined_results_csv):
    os.remove(refined_results_csv)

def save_refined_trial_callback(study, frozen_trial):
    row = {
        **base_best_params,
        **frozen_trial.params,
        "value": frozen_trial.value,
        "state": frozen_trial.state.name,
        "trial_number": frozen_trial.number,
    }
    pd.DataFrame([row]).to_csv(
        refined_results_csv,
        mode="a",
        header=not os.path.exists(refined_results_csv),
        index=False,
    )

In [23]:
def refined_objective(trial):
    model_name = base_best_params["model_name"]
    lr = base_best_params["lr"]
    wd = base_best_params["wd"]
    max_len = base_best_params["max_len"]
    dropout = base_best_params["dropout"]

    # kinda arbitrary range but whatever, take like +2 to prevent overfitting
    epochs = trial.suggest_int("epochs", base_best_params["epochs"], base_best_params["epochs"] + 2)
    alpha = trial.suggest_float("alpha", 0.1, 0.75)
    gamma = trial.suggest_float("gamma", 1.0, 4.0)

    trial_tokenizer = get_tokenizer(model_name)
    pcl_collation.MAX_LEN = max_len

    trial_train_loader = DataLoader(
        training_ds,
        batch_size=LOAD_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_wrapper(trial_tokenizer),
        pin_memory=PIN_MEMORY,
        num_workers=NUM_WORKERS,
    )

    trial_dev_loader = DataLoader(
        dev_ds,
        batch_size=LOAD_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_wrapper(trial_tokenizer),
        pin_memory=PIN_MEMORY,
        num_workers=NUM_WORKERS,
    )

    model = PCLModel(
        model_name,
        n_labels=NUM_LABELS,
        aux_dim=AUX_DIM,
        dropout=dropout,
        device=DEVICE,
        alpha=alpha,
        gamma=gamma,
    ).to(DEVICE)

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    print(
        "Refined config:",
        {
            "model_name": model_name,
            "lr": lr,
            "wd": wd,
            "max_len": max_len,
            "dropout": dropout,
            "epochs": epochs,
            "alpha": alpha,
            "gamma": gamma,
            "accum_steps": ACCUM_STEPS,
        },
    )

    try:
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            optim.zero_grad()

            for step, batch in enumerate(trial_train_loader):
                input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
                attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
                labels = batch["labels"].to(DEVICE, non_blocking=True)
                aux_features = batch["aux_features"].to(DEVICE, non_blocking=True) if "aux_features" in batch else None

                with torch.amp.autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, aux_features=aux_features)
                    loss = out["loss"] / ACCUM_STEPS

                scaler.scale(loss).backward()
                running_loss += out["loss"].item()

                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(trial_train_loader):
                    scaler.step(optim)
                    scaler.update()
                    optim.zero_grad()

            print(f"Epoch {epoch + 1} - Average Loss: {running_loss / len(trial_train_loader)}")

    except torch.cuda.OutOfMemoryError:
        print("OOM: pruning this refined trial")
        torch.cuda.empty_cache()
        del model
        raise optuna.TrialPruned()

    metrics = evaluate_dev(model, trial_dev_loader, DEVICE)
    print("Refined trial metrics:", str(metrics))
    del model
    torch.cuda.empty_cache()
    return metrics["bin_f1"]

In [24]:
torch.cuda.empty_cache()
N_REFINED_TRIALS = 30

In [25]:
refined_study = optuna.create_study(direction="maximize", study_name="pcl_refined_alpha_gamma")
refined_study.optimize(
    refined_objective,
    n_trials=N_REFINED_TRIALS,
    n_jobs=1,
    callbacks=[save_refined_trial_callback],
)

print(f"Saved per-trial refined results to {refined_results_csv}")

[I 2026-02-20 19:15:37,515] A new study created in memory with name: pcl_refined_alpha_gamma


Refined config: {'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'dropout': 0.0, 'epochs': 8, 'alpha': 0.5123281806171556, 'gamma': 1.9777445120477148, 'accum_steps': 3}
Epoch 1 - Average Loss: 0.27914532797254676
Epoch 2 - Average Loss: 0.21131467202834728
Epoch 3 - Average Loss: 0.1578908352072898
Epoch 4 - Average Loss: 0.09622289967600554
Epoch 5 - Average Loss: 0.0722072361306401
Epoch 6 - Average Loss: 0.04528325669181298
Epoch 7 - Average Loss: 0.03380149131341311
Epoch 8 - Average Loss: 0.023170600788985942


[I 2026-02-20 19:22:07,668] Trial 0 finished with value: 0.5356371490280778 and parameters: {'epochs': 8, 'alpha': 0.5123281806171556, 'gamma': 1.9777445120477148}. Best is trial 0 with value: 0.5356371490280778.


Refined trial metrics: {'bin_f1': 0.5356371490280778, 'multi_micro_f1': 0.1849514563106796, 'bin_ap': 0.5605124137975104}
Refined config: {'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'dropout': 0.0, 'epochs': 9, 'alpha': 0.1089485004859345, 'gamma': 1.046242460955649, 'accum_steps': 3}
Epoch 1 - Average Loss: 0.2854820530939819
Epoch 2 - Average Loss: 0.2037192308930467
Epoch 3 - Average Loss: 0.15032053261529654
Epoch 4 - Average Loss: 0.08270968926647493
Epoch 5 - Average Loss: 0.0505311590124398
Epoch 6 - Average Loss: 0.026182232125191718
Epoch 7 - Average Loss: 0.02509533234370716
Epoch 8 - Average Loss: 0.008485445968130468
Epoch 9 - Average Loss: 0.006782083567257443


[I 2026-02-20 19:29:24,504] Trial 1 finished with value: 0.4904632152588556 and parameters: {'epochs': 9, 'alpha': 0.1089485004859345, 'gamma': 1.046242460955649}. Best is trial 0 with value: 0.5356371490280778.


...

Refined trial metrics: {'bin_f1': 0.44686648501362397, 'multi_micro_f1': 0.1221230624706435, 'bin_ap': 0.4710848541918642}
Refined config: {'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'dropout': 0.0, 'epochs': 8, 'alpha': 0.3768297737711504, 'gamma': 1.2683121126139147, 'accum_steps': 3}
Epoch 1 - Average Loss: 0.28185316846343395
Epoch 2 - Average Loss: 0.20787899689535602
Epoch 3 - Average Loss: 0.15765148908244658
Epoch 4 - Average Loss: 0.09358404507062519
Epoch 5 - Average Loss: 0.059716460182605215
Epoch 6 - Average Loss: 0.03333989095121875
Epoch 7 - Average Loss: 0.021974738354693372
Epoch 8 - Average Loss: 0.019394810039231034


[I 2026-02-20 22:12:48,358] Trial 28 finished with value: 0.4306784660766962 and parameters: {'epochs': 8, 'alpha': 0.3768297737711504, 'gamma': 1.2683121126139147}. Best is trial 13 with value: 0.535796766743649.


Refined trial metrics: {'bin_f1': 0.4306784660766962, 'multi_micro_f1': 0.28414701042238066, 'bin_ap': 0.4533664197487083}
Refined config: {'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'dropout': 0.0, 'epochs': 9, 'alpha': 0.5401253549821036, 'gamma': 1.6566718234747242, 'accum_steps': 3}
Epoch 1 - Average Loss: 0.30179238476157755
Epoch 2 - Average Loss: 0.21659743936931472
Epoch 3 - Average Loss: 0.16505162806058202
Epoch 4 - Average Loss: 0.10243235465062725
Epoch 5 - Average Loss: 0.07504125241032901
Epoch 6 - Average Loss: 0.042846063056646115
Epoch 7 - Average Loss: 0.029854100570793598
Epoch 8 - Average Loss: 0.02934987919339786
Epoch 9 - Average Loss: 0.026720456204591285


[I 2026-02-20 22:19:57,278] Trial 29 finished with value: 0.523943661971831 and parameters: {'epochs': 9, 'alpha': 0.5401253549821036, 'gamma': 1.6566718234747242}. Best is trial 13 with value: 0.535796766743649.


Refined trial metrics: {'bin_f1': 0.523943661971831, 'multi_micro_f1': 0.2617893345085941, 'bin_ap': 0.5528884393684073}
Saved per-trial refined results to pcl_refined_hyperparam_results.csv


In [26]:
best_refined_trial = refined_study.best_trial
best_full_params = {
    **base_best_params,
    **best_refined_trial.params,
}

print("Best refined trial:")
print(f"  F1 (value): {best_refined_trial.value}")
print("  Params:")
for key, value in best_full_params.items():
    print(f"    {key}: {value}")

Best refined trial:
  F1 (value): 0.535796766743649
  Params:
    model_name: albert-base-v2
    lr: 1.4205892126825735e-05
    wd: 0.0008072537435151
    max_len: 128
    epochs: 7
    dropout: 0.0
    alpha: 0.4508917125675066
    gamma: 1.9241203656028858


In [27]:
refined_df = pd.read_csv(refined_results_csv)
best_logged_row = refined_df.loc[refined_df["value"].astype(float).idxmax()]
print("\nBest logged row from pcl_refined_hyperparam_results.csv:")
print(best_logged_row.to_dict())


Best logged row from pcl_refined_hyperparam_results.csv:
{'model_name': 'albert-base-v2', 'lr': 1.4205892126825735e-05, 'wd': 0.0008072537435151, 'max_len': 128, 'epochs': 7, 'dropout': 0.0, 'alpha': 0.4508917125675066, 'gamma': 1.9241203656028856, 'value': 0.535796766743649, 'state': 'COMPLETE', 'trial_number': 13}
